# Trial run: train the 0.8B student on a Kaggle T4

This notebook checks three things before a full run, on a few hundred rows:

1. Qwen3.5's Gated DeltaNet layers run forward and backward on a T4 (compute capability 7.5).
2. The loss stays finite in fp16. A T4 has no bf16, and `training/train.py` was written for bf16.
3. Seconds per step, to estimate the full run.

Settings: Accelerator = GPU T4 x2, Internet = on. Add the dataset with `sft_features/` and `sft_ghosts/` (each with `train.jsonl` and `val.jsonl`) as an input.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.__version__, torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0), 'bf16', torch.cuda.is_bf16_supported())

In [ ]:
# The versions in training/pyproject.toml.
!pip install -q "unsloth>=2026.9" "transformers>=5.5" "peft>=0.17"

In [ ]:
!git clone --depth 1 -b local/learning-poc https://github.com/dongliutony/decision-pacman.git /kaggle/working/decision-pacman
%cd /kaggle/working/decision-pacman

In [ ]:
import glob, os
# The Kaggle dataset uploaded from data/sft_p1.zip; change to your own <user>/<slug>.
DATASET = 'liudongtony/pacman-sft-p1'
# Look where Kaggle mounts inputs first. A newly added input can show up as an empty folder,
# so fall back to downloading the dataset (needs Internet on).
found = sorted(glob.glob('/kaggle/input/**/sft_*/train.jsonl', recursive=True))
if not found:
    import kagglehub
    path = kagglehub.dataset_download(DATASET)
    found = sorted(glob.glob(os.path.join(path, '**/sft_*/train.jsonl'), recursive=True))
print('\n'.join(found))
DATA = {os.path.basename(os.path.dirname(p)): os.path.dirname(p) for p in found}
assert {'sft_features', 'sft_ghosts'} <= DATA.keys(), DATA
DATA

## Trial: 640 rows, 20 steps of 32, evaluation every 10 steps

Look for: `projection check` below 0.5, a finite `loss` at step 20, `agree` rising from the `before training` line, and the seconds at step 20.

In [ ]:
!python training/train.py --data {DATA['sft_ghosts']} --out /kaggle/working/trial_ghosts \
    --train-limit 640 --val-limit 256 --epochs 1 --eval-every 10 --log-every 1

## If the trial passes: the full run

Two students from the same states and labels, one per GPU, in parallel: `sft_features` (nearest ghost only) and `sft_ghosts` (every ghost).
Each writes its LoRA adapter to `/kaggle/working/<name>/adapter`. Save the notebook version to keep `/kaggle/working`.

In [ ]:
import subprocess
procs = []
for gpu, name in enumerate(['sft_features', 'sft_ghosts']):
    log = open(f'/kaggle/working/{name}.log', 'w')
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    procs.append(subprocess.Popen(['python', 'training/train.py', '--data', DATA[name], '--out', f'/kaggle/working/{name}', '--epochs', '2', '--log-every', '10'],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
[p.pid for p in procs]

In [ ]:
# Rerun this cell to watch progress.
!tail -n 3 /kaggle/working/sft_features.log /kaggle/working/sft_ghosts.log